#Telco Customer Churn - CEO Interactive EDA Dashboard

7,043 customers · 1,869 churned (26.5%) · $1.66M annual revenue at risk

This notebook answers the 4 strategic questions a Telco CEO needs answered before investing in retention:

| # | Question | Section |
|---|----------|---------|
| Q1 | Who is churning? | Demographics, customer profiles, scatter explorer |
| Q2 | What drives churn? | Contract, payment, services - interactive segmentation |
| Q3 | When do customers leave? | Tenure lifecycle, survival curve, danger zones |
| Q4 | How much revenue is at risk? | Dollar impact by segment, what-if scenarios |

> Run all cells (`Kernel -> Restart & Run All`) to activate all interactive controls.

---
##Setup & data preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, HTML

try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, HBox, VBox, Layout
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("⚠ ipywidgets not installed. Run: pip install ipywidgets")
    print("  Falling back to static charts. Interactive controls require widgets.")

try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("⚠ plotly not installed. Run: pip install plotly")
    print("  Using matplotlib for all charts.")

#Professional color palette
PAL = {
    'churn': '#DC3545', 'stay': '#2F72D6', 'teal': '#0FA573',
    'amber': '#E8920D', 'violet': '#7048C6', 'slate': '#475978',
    'churn_light': '#FDECEE', 'stay_light': '#E8F0FC',
    'teal_light': '#E4F7EF', 'amber_light': '#FEF5E0',
    'bg': '#F8FAFC', 'card': '#FFFFFF', 'grid': '#EEF1F5',
    'text': '#0C1222', 'muted': '#64748B'
}

sns.set_style("white")
plt.rcParams.update({
    'figure.facecolor': PAL['bg'], 'axes.facecolor': PAL['card'],
    'axes.edgecolor': PAL['grid'], 'axes.grid': True,
    'grid.color': PAL['grid'], 'grid.linewidth': 0.5,
    'font.size': 11, 'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'figure.dpi': 120
})

print("✓ Setup complete")

In [ ]:
#Load and clean
#UPDATE THIS PATH to wherever your CSV lives
DATA_PATH = 'TelcoCustomerChurn_Assignment3.csv'

df = pd.read_csv(DATA_PATH)

#Clean missing values (4 strategies)
df['tenure'] = df['tenure'].fillna(df['tenure'].median())
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
tc_mask = df['TotalCharges'].isnull()
df.loc[tc_mask, 'TotalCharges'] = df.loc[tc_mask, 'tenure'] * df.loc[tc_mask, 'MonthlyCharges']
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['PaymentMethod'] = df['PaymentMethod'].fillna('Unknown')

#Collapse redundant categories
for col in ['OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies']:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

#Derived columns
df['Churn_int'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['SeniorLabel'] = df['SeniorCitizen'].map({0: 'Non-Senior', 1: 'Senior (65+)'})
df['tenure_bucket'] = pd.cut(df['tenure'], bins=[0,3,6,12,24,48,72],
    labels=['0-3 mo','3-6 mo','6-12 mo','12-24 mo','24-48 mo','48-72 mo'],
    include_lowest=True)

CHURN_RATE = df['Churn_int'].mean() * 100
TOTAL_RISK = df[df['Churn_int']==1]['MonthlyCharges'].sum() * 12

print(f"Loaded {len(df):,} customers")
print(f"Churned: {df['Churn_int'].sum():,} ({CHURN_RATE:.1f}%)")
print(f"Revenue at risk: ${TOTAL_RISK:,.0f}/year")

---
##Executive KPI summary

In [ ]:
churned_df = df[df['Churn_int'] == 1]
stayed_df = df[df['Churn_int'] == 0]

kpi_html = f"""
<div style="display:flex;gap:12px;flex-wrap:wrap;margin:10px 0 20px;">
  <div style="flex:1;min-width:150px;background:#fff;border:1px solid #D6DDE6;border-radius:10px;padding:16px 20px;">
    <div style="font-size:11px;color:#64748B;text-transform:uppercase;font-weight:600;letter-spacing:0.04em;">Total customers</div>
    <div style="font-size:28px;font-weight:700;color:#0C1222;">{len(df):,}</div>
    <div style="font-size:11px;color:#8896AB;">{df['Churn_int'].sum():,} churned</div>
  </div>
  <div style="flex:1;min-width:150px;background:#fff;border:1px solid #D6DDE6;border-radius:10px;padding:16px 20px;">
    <div style="font-size:11px;color:#64748B;text-transform:uppercase;font-weight:600;letter-spacing:0.04em;">Churn rate</div>
    <div style="font-size:28px;font-weight:700;color:#DC3545;">{CHURN_RATE:.1f}%</div>
    <div style="font-size:11px;color:#8896AB;">~1 in 4 customers</div>
  </div>
  <div style="flex:1;min-width:150px;background:#fff;border:1px solid #D6DDE6;border-radius:10px;padding:16px 20px;">
    <div style="font-size:11px;color:#64748B;text-transform:uppercase;font-weight:600;letter-spacing:0.04em;">Annual revenue at risk</div>
    <div style="font-size:28px;font-weight:700;color:#DC3545;">${TOTAL_RISK:,.0f}</div>
    <div style="font-size:11px;color:#8896AB;">From churned customers</div>
  </div>
  <div style="flex:1;min-width:150px;background:#fff;border:1px solid #D6DDE6;border-radius:10px;padding:16px 20px;">
    <div style="font-size:11px;color:#64748B;text-transform:uppercase;font-weight:600;letter-spacing:0.04em;">Avg tenure (churned)</div>
    <div style="font-size:28px;font-weight:700;color:#E8920D;">{churned_df['tenure'].mean():.0f} mo</div>
    <div style="font-size:11px;color:#8896AB;">vs {stayed_df['tenure'].mean():.0f} mo stayed</div>
  </div>
  <div style="flex:1;min-width:150px;background:#fff;border:1px solid #D6DDE6;border-radius:10px;padding:16px 20px;">
    <div style="font-size:11px;color:#64748B;text-transform:uppercase;font-weight:600;letter-spacing:0.04em;">Avg bill (churned)</div>
    <div style="font-size:28px;font-weight:700;color:#E8920D;">${churned_df['MonthlyCharges'].mean():.0f}/mo</div>
    <div style="font-size:11px;color:#8896AB;">vs ${stayed_df['MonthlyCharges'].mean():.0f} stayed</div>
  </div>
</div>
"""
display(HTML(kpi_html))

---
##Q1: Who is churning?
> Interactive controls: Select a demographic dimension and metric from the dropdowns.  
> Use the tenure and bill sliders to filter the scatter plot and isolate high-risk customer profiles.

In [ ]:
def plot_demographics(segment='SeniorLabel', metric='Churn Rate (%)'):
    col_map = {
        'Age Group': 'SeniorLabel', 'Partner Status': 'Partner',
        'Dependents': 'Dependents', 'Gender': 'gender'
    }
    col = col_map.get(segment, segment)

    grp = df.groupby(col).agg(
        total=('Churn_int','count'), churned=('Churn_int','sum'),
        rate=('Churn_int','mean'), avg_bill=('MonthlyCharges','mean'),
        avg_tenure=('tenure','mean')
    ).reset_index()
    grp['rate'] = grp['rate'] * 100

    metric_map = {'Churn Rate (%)': 'rate', 'Avg Monthly Bill ($)': 'avg_bill', 'Avg Tenure (mo)': 'avg_tenure'}
    m = metric_map[metric]

    fig, ax = plt.subplots(figsize=(10, max(3, len(grp)*1.2)))
    colors = [PAL['churn'] if r > CHURN_RATE else PAL['teal'] for r in grp['rate']]
    if m != 'rate':
        colors = [PAL['stay']] * len(grp)

    bars = ax.barh(grp[col], grp[m], color=colors, height=0.55, edgecolor='white')
    for bar, val, n in zip(bars, grp[m], grp['total']):
        suffix = '%' if m == 'rate' else ' mo' if m == 'avg_tenure' else ''
        prefix = '$' if m == 'avg_bill' else ''
        ax.text(bar.get_width() + (grp[m].max() * 0.02), bar.get_y() + bar.get_height()/2,
                f'{prefix}{val:.1f}{suffix}  (n={n:,})', va='center', fontsize=11, fontweight='bold')

    if m == 'rate':
        ax.axvline(CHURN_RATE, color=PAL['churn'], linestyle='--', linewidth=1, alpha=0.5,
                   label=f'Avg: {CHURN_RATE:.1f}%')
        ax.legend(fontsize=10)

    ax.set_title(f'{metric} by {segment}', loc='left', pad=12)
    ax.set_xlim(0, grp[m].max() * 1.35)
    sns.despine(ax=ax, left=True)
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    interact(plot_demographics,
             segment=widgets.Dropdown(options=['Age Group','Partner Status','Dependents','Gender'],
                                       value='Age Group', description='Segment:'),
             metric=widgets.Dropdown(options=['Churn Rate (%)','Avg Monthly Bill ($)','Avg Tenure (mo)'],
                                      value='Churn Rate (%)', description='Metric:'))
else:
    plot_demographics('Age Group', 'Churn Rate (%)')

###Scatter explorer - tenure vs monthly charges
> Drag the sliders to filter by minimum tenure and maximum bill. The filtered churn rate updates live.

In [ ]:
def plot_scatter(min_tenure=0, max_bill=120):
    filtered = df[(df['tenure'] >= min_tenure) & (df['MonthlyCharges'] <= max_bill)]
    filt_rate = filtered['Churn_int'].mean() * 100 if len(filtered) > 0 else 0
    n_filt = len(filtered)
    n_churned = filtered['Churn_int'].sum()

    sample = filtered.sample(min(2000, len(filtered)), random_state=42) if len(filtered) > 0 else filtered
    stayed = sample[sample['Churn_int'] == 0]
    churned = sample[sample['Churn_int'] == 1]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(stayed['tenure'], stayed['MonthlyCharges'], c=PAL['stay'],
               alpha=0.25, s=14, label=f'Stayed ({len(stayed)})', edgecolors='none')
    ax.scatter(churned['tenure'], churned['MonthlyCharges'], c=PAL['churn'],
               alpha=0.6, s=22, label=f'Churned ({len(churned)})', edgecolors='none')

    ax.set_xlabel('Tenure (months)', color=PAL['muted'])
    ax.set_ylabel('Monthly charges ($)', color=PAL['muted'])
    ax.set_title(f'Filtered: {n_filt:,} customers | Churn rate: {filt_rate:.1f}% ({n_churned:,} churned)',
                 loc='left', pad=12, fontsize=13)
    ax.legend(fontsize=10, loc='lower right')

    # Danger zone annotation
    if min_tenure < 12:
        ax.axvspan(min_tenure, min(12, 72), alpha=0.04, color=PAL['churn'])
        ax.text(min_tenure + 3, max_bill - 5, 'HIGH RISK\nZONE', fontsize=9,
                fontweight='bold', color=PAL['churn'], alpha=0.5)

    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    interact(plot_scatter,
             min_tenure=widgets.IntSlider(min=0, max=60, step=6, value=0,
                                           description='Min tenure:', style={'description_width':'100px'}),
             max_bill=widgets.IntSlider(min=20, max=120, step=10, value=120,
                                         description='Max bill ($):', style={'description_width':'100px'}))
else:
    plot_scatter(0, 120)

---
##Q2: What drives churn?
>Interactive controls: Select any variable from the dropdown to see its churn breakdown.  
> Toggle the metric to compare rates, bill sizes, or raw counts.

In [ ]:
def plot_driver(variable='Contract', metric='Churn Rate (%)'):
    grp = df.groupby(variable).agg(
        total=('Churn_int','count'), churned=('Churn_int','sum'),
        rate=('Churn_int','mean'), avg_bill=('MonthlyCharges','mean'),
        avg_tenure=('tenure','mean')
    ).reset_index()
    grp['rate'] = grp['rate'] * 100

    metric_map = {'Churn Rate (%)':'rate', 'Avg Monthly Bill ($)':'avg_bill',
                  '# Churned':'churned', 'Avg Tenure (mo)':'avg_tenure'}
    m = metric_map[metric]
    grp = grp.sort_values(m, ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(3.5, len(grp)*1.0)))

    if m == 'rate':
        colors = [PAL['churn'] if r > CHURN_RATE*1.3 else PAL['amber'] if r > CHURN_RATE else PAL['teal']
                  for r in grp['rate']]
    else:
        colors = [PAL['blue'] if i < len(grp)//2 else PAL['slate'] for i in range(len(grp))]
        colors = colors[::-1]

    bars = ax.barh(grp[variable], grp[m], color=colors, height=0.55, edgecolor='white')
    for bar, val, n in zip(bars, grp[m], grp['total']):
        suffix = '%' if m == 'rate' else ' mo' if m == 'avg_tenure' else ''
        prefix = '$' if m == 'avg_bill' else ''
        fmt_val = f'{prefix}{val:,.1f}{suffix}' if m not in ['churned'] else f'{int(val):,}'
        ax.text(bar.get_width() + grp[m].max()*0.02, bar.get_y()+bar.get_height()/2,
                f'{fmt_val}  (n={n:,})', va='center', fontsize=11, fontweight='bold')

    if m == 'rate':
        ax.axvline(CHURN_RATE, color=PAL['churn'], linestyle='--', linewidth=1, alpha=0.5,
                   label=f'Avg: {CHURN_RATE:.1f}%')
        ax.legend(fontsize=10)

    ax.set_title(f'{metric} by {variable}', loc='left', pad=12)
    ax.set_xlim(0, grp[m].max() * 1.4)
    sns.despine(ax=ax, left=True)
    plt.tight_layout()
    plt.show()

DRIVER_OPTIONS = ['Contract','InternetService','PaymentMethod','PaperlessBilling',
                  'TechSupport','OnlineSecurity','OnlineBackup','DeviceProtection',
                  'StreamingTV','StreamingMovies','Partner','Dependents','SeniorLabel']

if HAS_WIDGETS:
    interact(plot_driver,
             variable=widgets.Dropdown(options=DRIVER_OPTIONS, value='Contract', description='Variable:'),
             metric=widgets.Dropdown(options=['Churn Rate (%)','Avg Monthly Bill ($)',
                                               '# Churned','Avg Tenure (mo)'],
                                      value='Churn Rate (%)', description='Metric:'))
else:
    plot_driver('Contract', 'Churn Rate (%)')

###Service add-on impact - protection vs entertainment

In [ ]:
services = ['OnlineSecurity','TechSupport','OnlineBackup',
            'DeviceProtection','StreamingTV','StreamingMovies']
svc_data = []
for svc in services:
    no_rate = df[df[svc]=='No']['Churn_int'].mean() * 100
    yes_rate = df[df[svc]=='Yes']['Churn_int'].mean() * 100
    svc_data.append({'service': svc, 'Without': no_rate, 'With': yes_rate, 'delta': no_rate - yes_rate})

svc_df = pd.DataFrame(svc_data).sort_values('delta', ascending=True)

fig, ax = plt.subplots(figsize=(12, 5))
y = np.arange(len(svc_df))
ax.barh(y - 0.18, svc_df['Without'], height=0.34, color=PAL['churn'], alpha=0.75, label='Without service')
ax.barh(y + 0.18, svc_df['With'], height=0.34, color=PAL['teal'], alpha=0.75, label='With service')

for i, (_, row) in enumerate(svc_df.iterrows()):
    sign = '−' if row['delta'] > 0 else '+'
    color = PAL['teal'] if row['delta'] > 0 else PAL['churn']
    ax.text(max(row['Without'], row['With']) + 1.5, i,
            f'{sign}{abs(row["delta"]):.1f}pp', va='center', fontsize=11,
            fontweight='bold', color=color)

ax.set_yticks(y)
ax.set_yticklabels(svc_df['service'], fontsize=12)
ax.set_xlabel('Churn rate (%)', color=PAL['muted'])
ax.set_title('Service add-on impact on churn rate', loc='left', pad=12)
ax.axvline(CHURN_RATE, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)
ax.legend(fontsize=10, loc='lower right')
sns.despine(ax=ax, left=True)
plt.tight_layout()
plt.show()

print("\n-> Online security and tech support cut churn by ~16pp each.")
print("-> Streaming services slightly INCREASE churn (correlated with fiber optic).")

---
##Q3: When do customers leave?
> Interactive controls: Toggle between the survival curve, tenure histogram, or both.  
> Adjust the danger zone threshold to see what percentage of churn happens before that cutoff.

In [ ]:
def plot_tenure(view='Both', danger_cutoff=6):
    tenure_rates = df.groupby('tenure_bucket', observed=True)['Churn_int'].mean().reset_index()
    tenure_rates['rate'] = tenure_rates['Churn_int'] * 100

    if view in ['Survival Curve', 'Both']:
        fig1, ax1 = plt.subplots(figsize=(12, 5))
        x = range(len(tenure_rates))
        ax1.fill_between(x, tenure_rates['rate'], alpha=0.12, color=PAL['churn'])
        ax1.plot(x, tenure_rates['rate'], color=PAL['churn'], linewidth=3,
                 marker='o', markersize=10, markerfacecolor='white',
                 markeredgecolor=PAL['churn'], markeredgewidth=2.5)
        for i, (_, row) in enumerate(tenure_rates.iterrows()):
            ax1.annotate(f'{row["rate"]:.1f}%', (i, row['rate']),
                         textcoords="offset points", xytext=(0, 16),
                         ha='center', fontsize=12, fontweight='bold', color=PAL['churn'])
        ax1.set_xticks(x)
        ax1.set_xticklabels(tenure_rates['tenure_bucket'], fontsize=10)
        ax1.set_ylabel('Churn rate (%)', color=PAL['muted'])
        ax1.set_title('Churn rate by tenure - the survival curve', loc='left', pad=12)
        ax1.set_ylim(0, 65)
        sns.despine(ax=ax1)
        plt.tight_layout()
        plt.show()

    if view in ['Histogram', 'Both']:
        bins = list(range(0, 78, 6))
        fig2, ax2 = plt.subplots(figsize=(12, 5))
        stayed_t = df[df['Churn_int']==0]['tenure']
        churned_t = df[df['Churn_int']==1]['tenure']
        ax2.hist([stayed_t, churned_t], bins=bins, stacked=True,
                 color=[PAL['stay'], PAL['churn']], label=['Stayed','Churned'],
                 edgecolor='white', linewidth=0.5)
        ax2.axvspan(0, danger_cutoff, alpha=0.08, color=PAL['churn'])

        # Calculate % of churn before cutoff
        early_churn = df[(df['Churn_int']==1) & (df['tenure'] <= danger_cutoff)].shape[0]
        total_churn = df['Churn_int'].sum()
        pct_early = early_churn / total_churn * 100

        ax2.text(danger_cutoff/2, ax2.get_ylim()[1]*0.85,
                 f'DANGER ZONE\n{pct_early:.0f}% of all churn\nhappens here',
                 ha='center', fontsize=10, fontweight='bold', color=PAL['churn'], alpha=0.7)

        ax2.axvline(danger_cutoff, color=PAL['churn'], linestyle=':', linewidth=1.5, alpha=0.6)
        ax2.set_xlabel('Tenure (months)', color=PAL['muted'])
        ax2.set_ylabel('Number of customers', color=PAL['muted'])
        ax2.set_title(f'Tenure distribution - danger zone ≤ {danger_cutoff} months', loc='left', pad=12)
        ax2.legend(fontsize=10)
        sns.despine(ax=ax2)
        plt.tight_layout()
        plt.show()

        print(f"\n-> {early_churn:,} of {total_churn:,} churned customers ({pct_early:.1f}%) left within {danger_cutoff} months.")

if HAS_WIDGETS:
    interact(plot_tenure,
             view=widgets.ToggleButtons(options=['Survival Curve','Histogram','Both'],
                                         value='Both', description='View:'),
             danger_cutoff=widgets.IntSlider(min=3, max=24, step=3, value=6,
                                              description='Danger zone:', style={'description_width':'110px'}))
else:
    plot_tenure('Both', 6)

---
Q4: How much revenue is at risk?
> Interactive controls: Break down revenue by contract type or internet service.  
> Use the churn reduction slider to model what-if scenarios: how much revenue is saved if churn drops by X percentage points?

In [ ]:
def plot_revenue(breakdown='Contract', churn_reduction_pp=5):
    churned_df = df[df['Churn_int'] == 1]
    rev = churned_df.groupby(breakdown)['MonthlyCharges'].agg(['sum','count']).reset_index()
    rev['annual'] = rev['sum'] * 12
    rev = rev.sort_values('annual', ascending=True)

    total_risk = rev['annual'].sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(rev)*1.2)),
                              gridspec_kw={'width_ratios': [1.4, 1]})

    # Left: horizontal bars
    ax = axes[0]
    palette = [PAL['teal'], PAL['amber'], PAL['churn']]
    if len(rev) > 3:
        palette = [PAL['teal']] + [PAL['amber']]*(len(rev)-2) + [PAL['churn']]

    bars = ax.barh(rev[breakdown], rev['annual'], color=palette[-len(rev):],
                    height=0.5, edgecolor='white')
    for bar, val, n in zip(bars, rev['annual'], rev['count']):
        label = f'${val:,.0f}' if val < 1e6 else f'${val/1e6:.2f}M'
        ax.text(bar.get_width() + total_risk*0.02, bar.get_y()+bar.get_height()/2,
                f'{label}  ({n:,} customers)', va='center', fontsize=11, fontweight='bold')
    ax.set_xlabel('Annual revenue at risk ($)', color=PAL['muted'])
    ax.set_title(f'Revenue at risk by {breakdown.lower()}', loc='left', pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K' if x < 1e6 else f'${x/1e6:.1f}M'))
    ax.set_xlim(0, rev['annual'].max() * 1.5)
    sns.despine(ax=ax, left=True)

    # Right: what-if scenario
    ax2 = axes[1]
    ax2.axis('off')

    avg_churned_bill = churned_df['MonthlyCharges'].mean()
    customers_saved = int(len(df) * churn_reduction_pp / 100)
    revenue_saved = customers_saved * avg_churned_bill * 12
    new_churn_rate = CHURN_RATE - churn_reduction_pp
    top_share = rev['annual'].iloc[-1] / total_risk * 100

    info = [
        ('Total annual risk', f'${total_risk:,.0f}', PAL['churn']),
        (f'Top segment share', f'{top_share:.0f}%', PAL['churn']),
        (f'If churn drops {churn_reduction_pp}pp', f'~{customers_saved:,} customers saved', PAL['amber']),
        (f'Revenue recovered', f'${revenue_saved:,.0f}/yr', PAL['teal']),
        (f'New churn rate', f'{new_churn_rate:.1f}%', PAL['teal']),
    ]

    for i, (label, value, color) in enumerate(info):
        y = 0.85 - i * 0.18
        ax2.text(0.08, y, label, fontsize=11, color=PAL['muted'], transform=ax2.transAxes, fontweight=500)
        ax2.text(0.08, y - 0.07, value, fontsize=18, color=color, transform=ax2.transAxes, fontweight='bold')

    ax2.set_title('What-if scenario', loc='left', pad=12)
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    interact(plot_revenue,
             breakdown=widgets.Dropdown(options=['Contract','InternetService','PaymentMethod'],
                                         value='Contract', description='Break down by:'),
             churn_reduction_pp=widgets.IntSlider(min=1, max=15, step=1, value=5,
                                                    description='Reduce churn by:',
                                                    style={'description_width':'130px'}))
else:
    plot_revenue('Contract', 5)

---
##Plotly interactive scatter (if plotly installed)
> This creates a fully interactive scatter with hover tooltips showing customer details.  
> Requires `pip install plotly`.

In [ ]:
if HAS_PLOTLY:
    sample = df.sample(min(2000, len(df)), random_state=42)
    sample['Status'] = sample['Churn_int'].map({0: 'Stayed', 1: 'Churned'})

    fig = px.scatter(sample, x='tenure', y='MonthlyCharges', color='Status',
                     color_discrete_map={'Stayed': PAL['stay'], 'Churned': PAL['churn']},
                     opacity=0.5, size_max=8,
                     hover_data=['Contract','InternetService','PaymentMethod','TotalCharges'],
                     title='Interactive scatter - hover for customer details',
                     labels={'tenure':'Tenure (months)','MonthlyCharges':'Monthly charges ($)'})
    fig.update_layout(height=500, template='plotly_white',
                      font=dict(family='DM Sans, sans-serif'))
    fig.show()
else:
    print("Plotly not installed. Run: pip install plotly")
    print("Then restart the kernel and re-run this cell.")

---
##Key findings summary

| Question | Finding | Number |
|----------|---------|--------|
| Q1: Who?| Seniors, customers without partners/dependents churn most | 41.7% senior churn rate |
| Q2: What?| Month-to-month contracts and electronic check payments are the top drivers | 42.7% MTM churn vs 2.8% two-year |
| Q3: When?| The first 6 months is the danger zone - most churn is concentrated here | 56.1% churn rate in first 3 months |
| Q4: How much?| $1.45M/year at risk from month-to-month alone (87% of total) | $1.66M total annual risk |

### Recommended actions
1. Convert month-to-month -> annual: Offer incentives at the 6-month mark. Even 10% conversion ≈ $145K/year recovered.
2. Bundle tech support into fiber: Fiber customers churn at 42% and pay the most. Tech support cuts churn by 16pp.
3. Early intervention program: Target customers in their first 90 days with onboarding, check-ins, and loyalty offers.
